## **0. SETUP: IMPORT LIBRARIES AND FILES**

In [ ]:
%pip install bertopic
%pip install spacy
%pip install hdbscan
%pip install sentence_transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 22.9 MB/s  0:00:00 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.3.4-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached pandas-2.3.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached scikit_learn-1.7.2-cp313-cp313-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached scipy-1.16.3-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.5.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached torch-2.9.0-cp313-none-macosx_11_0_arm64.whl.m

In [5]:
import pandas as pd
import numpy as np
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
import spacy
import spacy.cli
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from bertopic.vectorizers import ClassTfidfTransformer
from multiprocessing import cpu_count


# Ensure the language model is downloaded
spacy.cli.download("en_core_web_sm")
# Load spaCy English model
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "textcat"])
nlp.max_length = 3000000
# Load all_files database.
df = pd.read_csv("Text Files/Text Files/duplicate_handling/all_files.csv")
texts = df["text"]
doc_list = list(texts)
len(doc_list)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 65.8 MB/s  0:00:00m0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


7717

## **1. CLEAN AND PREPROCESS**

#### a. Remove all template text

In [6]:
templatePhrases = []
templatePhrases.append("Include the chapter of the model curriculum, the page number, and line number(s) to ensure that the California Department of Education and Instructional Quality Commission can reference the content of the document when reviewing your comments. Please email this document as a Word document to ethnicstudies@cde.ca.gov. You may contact Kenneth McDonald, Education Programs Consultant, at kmcdonal@cde.ca.gov with any questions regarding this template or the public input process.")
templatePhrases.append("Your Name and Affiliation")
templatePhrases.append("Comment (include page and line numbers where applicable)")
templatePhrases.append("(Download and use to provide specific recommendations)")
templatePhrases.append("2020 Ethnic Studies Model Curriculum May 2019 Draft")
templatePhrases.append("Public Input Template�")
templatePhrases.append('"General" if your comment isn\'t about one')
templatePhrases.append("Curriculum [Enter the Chapter Number here,")
templatePhrases.append('or just "General" for a comment that applies to the')
templatePhrases.append("entire document.]")
templatePhrases.append("[Enter the agency, organization, or business that you represent, if applicable.]")
templatePhrases.append("[Include the page and line number(s) here�Write your comment here]")

def removeTemplatePhrases(text, phraseList):
    result = text
    for phrase in phraseList:
        result = result.replace(phrase, '')
    return result

cleanedDocs = [removeTemplatePhrases(doc, templatePhrases) for doc in doc_list]

#### b. Lemmatize docs
##### *This cell takes 15-20 minutes to run*

In [7]:
lemmatized_docs = []
for spacy_doc in nlp.pipe(cleanedDocs, batch_size=50, n_process=cpu_count()):
    lemmas = [token.lemma_ for token in spacy_doc if not token.is_punct and not token.is_space]
    lemmatized_docs.append(" ".join(lemmas))
docs = lemmatized_docs

## **2. FIT AND TRAIN MODEL**
#### a. Pre-calculate embeddings to speed up tuning later
##### *This cell takes the longest time to run*

In [9]:
embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device="mps")

In [10]:
batch_size = 32
embeddings = embedding_model.encode(
    docs,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches: 100%|██████████| 242/242 [14:30<00:00,  3.60s/it]


In [11]:
representation_model = KeyBERTInspired()
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

topic_model = BERTopic(representation_model=representation_model,
                       embedding_model=embedding_model,
                       hdbscan_model=hdbscan_model,
                       ctfidf_model=ctfidf_model,
                       language="english",
                       calculate_probabilities=False, verbose=True)

In [12]:
topics, probs = topic_model.fit_transform(docs, embeddings)
freq = topic_model.get_topic_info()

2025-11-05 13:57:40,428 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-11-05 13:57:57,628 - BERTopic - Dimensionality - Completed ✓
2025-11-05 13:57:57,630 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-05 13:57:57,846 - BERTopic - Cluster - Completed ✓
2025-11-05 13:57:57,852 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-11-05 13:58:14,277 - BERTopic - Representation - Completed ✓


In [14]:
freq.to_csv("secondFinetunedTopics.csv", index=False)

